# Ford Smoke Test in Google Colab

This notebook is a lightweight smoke-test workflow for the benchmark repository. It proves that the repository can be opened from GitHub in Colab, can read a small Ford subset from Google Drive, can run the existing Isolation Forest baseline, and can save precision, recall, and F1 outputs.

## How to open in Colab

1. In Colab, choose **File → Open notebook → GitHub**.
2. Paste the GitHub repository URL and open `colab/01_ford_smoke_test.ipynb`.
3. If the repository is private, authorize Colab/GitHub access or clone with a temporary GitHub token stored only in the current Colab session. Do **not** write tokens into notebook cells, Git, or Google Drive.

## Runtime and execution

Isolation Forest does not require a GPU. If you still want to select one for consistency, choose **Runtime → Change runtime type → GPU**, then run **Runtime → Run all**.

## Required Google Drive dataset layout

Store real Ford data in Google Drive without committing or copying it into the repository:

```text
/content/drive/MyDrive/anomaly_benchmark_data/ford/Train/accept/Sweep_trans_0_accept.csv
/content/drive/MyDrive/anomaly_benchmark_data/ford/Train/reject/Sweep_trans_0_reject.csv
```

Outputs are copied to:

```text
/content/drive/MyDrive/anomaly_benchmark_results/ford_smoke_test/
```


In [ ]:
from pathlib import Path
import os
import platform
import sys

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
RANDOM_SEED = 42
DATASET_ROOT = Path('/content/drive/MyDrive/anomaly_benchmark_data')
RESULT_ROOT = Path('/content/drive/MyDrive/anomaly_benchmark_results/ford_smoke_test')
REPO_URL = ''  # Optional: set only if this notebook is not already running inside a cloned/opened repo.
BRANCH = 'main'

print(f'In Colab: {IN_COLAB}')
print(sys.version)
print(platform.platform())


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Colab; skipping Google Drive mount.')


## Repository setup

If this notebook was opened directly from the repository in Colab, the next cell installs dependencies from the current working tree. If you copied the notebook elsewhere, set `REPO_URL` above. For a private repository, authenticate through Colab/GitHub or use a token only as an ephemeral runtime secret; never commit credentials.


In [ ]:
import subprocess

cwd = Path.cwd()
if not (cwd / 'pyproject.toml').exists():
    repo_dir = Path('/content/INFORMS_FORD')
    if not repo_dir.exists():
        if not REPO_URL:
            raise RuntimeError('Set REPO_URL if the notebook is not already running inside the repository.')
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    repo_dir = cwd
print(f'Repository directory: {Path.cwd()}')


In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .


In [ ]:
import json
import random
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from benchmark.datasets.ford import FordDatasetConfig, FordDatasetLoader
from benchmark.metrics.classification import evaluate_binary
from benchmark.metrics.thresholding import select_threshold
from benchmark.models.isolation_forest import IsolationForestDetector

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('benchmark imports succeeded')


## Load a small Ford subset

This smoke test uses explicit small file-index lists to avoid reading the full dataset. Adjust the indices only after confirming those files exist in Google Drive. The loader still fits preprocessing on accepted training files only, uses accepted validation scores for threshold selection, and never uses rejected/test labels for training or threshold selection.


In [ ]:
FORD_ROOT = DATASET_ROOT / 'ford' / 'Train'
ACCEPTED_DIR = FORD_ROOT / 'accept'
REJECTED_DIR = FORD_ROOT / 'reject'
SMOKE_TRAIN_INDICES = [0, 1]
SMOKE_TEST_INDICES = [0]

for required in [ACCEPTED_DIR, REJECTED_DIR]:
    if not required.exists():
        raise FileNotFoundError(f'Missing required Google Drive dataset directory: {required}')

feature_names = [f'feature_{i}' for i in range(48)]
config = {
    'name': 'ford',
    'accepted_dir': str(ACCEPTED_DIR),
    'rejected_dir': str(REJECTED_DIR),
    'accepted_file_pattern': 'Sweep_trans_{index}_accept.csv',
    'rejected_file_pattern': 'Sweep_trans_{index}_reject.csv',
    'feature_columns': feature_names,
    'label_column': 'feature_55',
    'normal_label': 0,
    'anomaly_label': 1,
    'train_indices': SMOKE_TRAIN_INDICES,
    'test_indices': SMOKE_TEST_INDICES,
    'validation_fraction': 0.5,
    'split_by_file': True,
    'random_seed': RANDOM_SEED,
    'missing_values': {'strategy': 'median'},
    'clipping': {'enabled': True, 'lower_quantile': 0.001, 'upper_quantile': 0.999},
    'scaling': {'method': 'standard'},
    'windowing': {'lookback': 10, 'stride': 1, 'representation': 'full_window_flattened'},
    'downsampling': {'every_n_rows': 1},
}

loader = FordDatasetLoader(FordDatasetConfig.from_dict(config))
accepted_files = loader.discover_files('accept')
rejected_files = loader.discover_files('reject')
print('Accepted files:', [p.name for _, p in accepted_files])
print('Rejected files:', [p.name for _, p in rejected_files])

raw_missing_counts = {}
for _, path in accepted_files + rejected_files:
    df_head = pd.read_csv(path, usecols=lambda c: c in set(feature_names + ['feature_55']))
    raw_missing_counts[path.name] = int(df_head.isna().sum().sum())

bundle = loader.load()
print('X_train:', bundle.X_train.shape)
print('X_validation:', bundle.X_validation.shape)
print('X_test:', bundle.X_test.shape)
print('Feature count:', len(bundle.feature_names))
print('Feature names:', bundle.feature_names)
print('Validation class distribution:', dict(zip(*np.unique(bundle.y_validation, return_counts=True))))
print('Test class distribution:', dict(zip(*np.unique(bundle.y_test, return_counts=True))))
print('Raw missing value counts:', raw_missing_counts)


## Run existing lightweight baseline

The threshold is selected from validation scores only using the repository thresholding utility. Test labels are used only once, for final metric computation.


In [ ]:
model = IsolationForestDetector(n_estimators=25, contamination='auto', random_state=RANDOM_SEED)
model.fit(bundle.X_train)
validation_scores = model.score_samples(bundle.X_validation)
test_scores = model.score_samples(bundle.X_test)
threshold = select_threshold(validation_scores, method='percentile', percentile=95.0)
metrics = evaluate_binary(bundle.y_test, test_scores, threshold.threshold)

smoke_metrics = {
    'schema_version': 'ford_smoke_test_v1',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'random_seed': RANDOM_SEED,
    'dataset': 'ford',
    'model': 'isolation_forest',
    'train_indices': SMOKE_TRAIN_INDICES,
    'test_indices': SMOKE_TEST_INDICES,
    'threshold_method': threshold.method,
    'threshold_value': threshold.threshold,
    'precision': metrics['precision'],
    'recall': metrics['recall'],
    'f1': metrics['f1'],
    'validation_shape': list(bundle.X_validation.shape),
    'test_shape': list(bundle.X_test.shape),
    'test_class_distribution': {str(k): int(v) for k, v in zip(*np.unique(bundle.y_test, return_counts=True))},
    'warnings': metrics.get('warnings', []) + bundle.warnings,
}
print(json.dumps(smoke_metrics, indent=2))


In [ ]:
local_output = Path('outputs') / 'ford_smoke_test'
local_output.mkdir(parents=True, exist_ok=True)
json_path = local_output / 'ford_smoke_test_metrics.json'
csv_path = local_output / 'ford_smoke_test_metrics.csv'
json_path.write_text(json.dumps(smoke_metrics, indent=2))
pd.DataFrame([smoke_metrics]).to_csv(csv_path, index=False)

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
for path in [json_path, csv_path]:
    shutil.copy2(path, RESULT_ROOT / path.name)

print(f'Local JSON: {json_path}')
print(f'Local CSV: {csv_path}')
print(f'Copied outputs to: {RESULT_ROOT}')


In [ ]:
pd.read_csv(csv_path)[['dataset', 'model', 'precision', 'recall', 'f1', 'threshold_method']]
